# Comparison with existing identifier mappers (Figure 4d)

This notebook provides a **capability-first** comparison between IDTrack and common point-in-time identifier mappers.

## Rationale (what this is proving)

Most identifier mappers are designed for a *single* implicit time point ("current") and return a best-effort mapping.
That is useful for convenience, but it is insufficient for **reproducible atlas-scale integration**, where you need to:

- Specify **which Ensembl history window** is allowed to influence results (snapshot boundary)
- Convert across releases as an explicit **time axis** (time travel)
- Treat **1→n** outcomes as legitimate, reportable results (not bugs to hide)
- Preserve enough metadata to make conversions auditable and rerunnable

This notebook markets IDTrack by contrasting those *reproducibility controls* against widely used point-in-time tools.

## Scope rules (important)

- This is **not** an accuracy benchmark.
- The demo queries are intentionally small and safe for public APIs.
- The comparison is framed around **capabilities** and **failure/ambiguity semantics**, not speed claims.

Outputs:
- `idtrack-manuscript/figures/fig_tool_comparison_matrix.pdf`
- Optional extra figures written into `idtrack-manuscript/figures/` (see plotting cells)

Caching:
- External-mapper query results are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/comparison/`.
- Re-running the notebook should not re-query external APIs unless you delete the cache.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    apply_rcparams,
    experiments_cache_dir,
    idtrack_cache_dir,
    load_rcparams,
    manuscript_palette,
    manuscript_figures_dir,
    read_pickle,
    write_pickle,
)

try:
    apply_rcparams(load_rcparams())
except Exception as e:  # noqa: S110
    print('Warning: could not apply shared rcParams:', e)

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
CACHE_DIR = experiments_cache_dir(REPO_ROOT, experiment='comparison')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configure comparison --------------------

# External mapper methods (optional deps)
METHODS = ['pybiomart', 'mygene', 'gprofiler', 'gget']

# Demonstration query set (small on purpose; safe for public APIs)
# - includes a version-suffixed Ensembl ID (to exercise version stripping)
# - includes an invalid ID (to guarantee at least one 1→0 outcome)
QUERY_IDS = [
    'ENSG00000139618',  # BRCA2
    'ENSG00000141510',  # TP53
    'ENSG00000157764',  # BRAF
    'ENSG00000121879',  # KRAS
    'ENSG00000171862',  # PTEN
    'ENSG00000136997',  # MYC
    'ENSG00000146648',  # EGFR
    'ENSG00000141510.18',  # TP53 (versioned)
    'ENSG_DOES_NOT_EXIST',  # negative control
]

INPUT_DB = 'ensembl_gene'
SPECIES = 'human'

# Targets to demo (marketing focus: HGNC / UniProt)
TARGETS = [
    {'label': 'HGNC symbols', 'output_db': 'HGNC Symbol'},
    {'label': 'UniProt accessions', 'output_db': 'UniProtKB/Swiss-Prot'},
]

# pybiomart only: pin an explicit historical release.
# (Other services usually do not expose this as a stable control.)
PYBIOMART_RELEASE = 107

print('N queries:', len(QUERY_IDS))
print('Targets:', [t['output_db'] for t in TARGETS])


In [ ]:
# -------------------- External mapper availability --------------------

import idtrack._external_mappers as ext

status = ext.check_optional_dependencies(warn=True)

pd.DataFrame([{'dependency': k, 'installed': v} for k, v in status.items()]).sort_values('dependency').reset_index(drop=True)


In [ ]:
# -------------------- Run external mappers (cached) --------------------

import idtrack._external_mappers as ext


def _safe_tag(s: str) -> str:
    return ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(s))


def _cache_path(method: str, output_db: str) -> Path:
    tag = f"extmap_{method}_in{INPUT_DB}_out{_safe_tag(output_db)}_species{SPECIES}_pybiomart{PYBIOMART_RELEASE}.pickle"
    return CACHE_DIR / tag


def _normalize(df: pd.DataFrame | None) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=['input_id', 'output_id', 'mapping'])
    x = df.copy()
    # Normalize to a stable schema for downstream counting + exports
    cols = [
        c
        for c in ['input_id', 'output_id', 'mapping', 'method', 'input_db', 'output_db', 'release_used']
        if c in x.columns
    ]
    return x[cols].copy()


def _counts_1_to_0_1_to_1_1_to_n(df: pd.DataFrame, inputs: list[str]) -> dict[str, int]:
    if df is None or df.empty:
        return {'1→0': len(set(inputs)), '1→1': 0, '1→n': 0}
    per = df.drop_duplicates('input_id')
    counts = per['mapping'].value_counts().to_dict()
    return {
        '1→0': int(counts.get('1:0', 0)),
        '1→1': int(counts.get('1:1', 0)),
        '1→n': int(counts.get('1:n', 0)),
    }


results: dict[tuple[str, str], pd.DataFrame] = {}
errors: list[dict] = []

for target in TARGETS:
    output_db = str(target['output_db'])
    for method in METHODS:
        p = _cache_path(method, output_db)
        if p.exists():
            df = read_pickle(p)
            results[(output_db, method)] = _normalize(df)
            print('Loaded:', p.name)
            continue

        try:
            kwargs = {
                'ids': QUERY_IDS,
                'input_db': INPUT_DB,
                'output_db': output_db,
                'method': method,
                'species': SPECIES,
                'chunk_size': 200,
                'pause': 0.1,
                'verbose': 2,
            }
            if method == 'pybiomart':
                kwargs['release_for_pybiomart'] = PYBIOMART_RELEASE

            df = ext.convert_ids(**kwargs)
            df = _normalize(df)
            write_pickle(df, p)
            results[(output_db, method)] = df
            print('Saved:', p.name, 'rows', len(df))
        except Exception as e:
            errors.append({'output_db': output_db, 'method': method, 'error': repr(e)})
            print('Skip', output_db, method, '->', repr(e))

# Summary table: outcome profile per method AND per target
summary_rows = []
for target in TARGETS:
    output_db = str(target['output_db'])
    for method in METHODS:
        df = results.get((output_db, method), pd.DataFrame())
        counts = _counts_1_to_0_1_to_1_1_to_n(df, QUERY_IDS)
        summary_rows.append(
            {
                'output_db': output_db,
                'method': method,
                'n_inputs': len(set(QUERY_IDS)),
                **counts,
            }
        )

summary = pd.DataFrame(summary_rows).sort_values(['output_db', 'method']).reset_index(drop=True)

summary


In [ ]:
# -------------------- Notes --------------------

# This notebook intentionally avoids running IDTrack conversions because graph loading can be memory-intensive.
# For IDTrack outcome-profile panels and drift diagnostics, see:
# - `random_data.ipynb`
# - `hlca_manuscript_figures.ipynb`


In [ ]:
# -------------------- Build capability matrix (Figure 4d) --------------------

capabilities = [
    'Target historical release',
    'Snapshot-bounded reproducibility knob',
    'Explicit 1→0 / 1→1 / 1→n semantics',
    'Audit payload / explainability',
    'Explicit external allowlist (YAML contract)',
    'Assembly-aware build axis',
]

tools = ['IDTrack', 'pybiomart', 'mygene', 'g:Profiler', 'gget']

# Conservative, capability-first matrix.
cap = pd.DataFrame(False, index=tools, columns=capabilities)
cap.loc['IDTrack', :] = [True, True, True, True, True, True]
cap.loc['pybiomart', 'Target historical release'] = True

mat = cap.astype(int)

fig = plt.figure(figsize=(12, 4.8), constrained_layout=True)

gs = fig.add_gridspec(1, 2, width_ratios=[1.45, 1.0])
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])

if sns is not None:
    sns.heatmap(
        mat,
        ax=ax0,
        cmap=['#FFFFFF', MANUSCRIPT_COLORS['1→1']],
        cbar=False,
        linewidths=0.5,
        linecolor=MANUSCRIPT_COLORS['grid'],
    )
else:
    ax0.imshow(mat.values)

ax0.set_title('Capability matrix (not an accuracy benchmark)')
ax0.set_xlabel('')
ax0.set_ylabel('')
ax0.set_xticklabels(ax0.get_xticklabels(), rotation=25, ha='right')

# Demo outcome profile for the external mappers (if available)
DEMO_OUTPUT_DB = 'HGNC Symbol'
if 'summary' in globals() and not summary.empty and (summary['output_db'] == DEMO_OUTPUT_DB).any():
    s = summary[summary['output_db'] == DEMO_OUTPUT_DB].set_index('method')[['1→0', '1→1', '1→n']]
    s_norm = s.div(s.sum(axis=1), axis=0)
    s_norm.plot(
        kind='bar',
        stacked=True,
        ax=ax1,
        color=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
    )
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Fraction of queries')
    ax1.set_title(f'Outcome profile (demo; target: {DEMO_OUTPUT_DB})')
    ax1.legend(['1→0', '1→1', '1→n'], loc='upper right')
else:
    ax1.axis('off')
    ax1.text(0.5, 0.5, 'External-mapper demo not available', ha='center', va='center')

out_fig = MANUSCRIPT_FIGURES / 'fig_tool_comparison_matrix.pdf'
fig.savefig(out_fig, bbox_inches='tight')
print('Saved:', out_fig)
